In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS shopsphere.quarantine;
CREATE SCHEMA IF NOT EXISTS shopsphere.silver;

CREATE TABLE IF NOT EXISTS shopsphere.silver.customers(
    customer_id int,
    first_name string,
    last_name string,
    email string,
    phone string,
    city string,
    state string,
    country string,
    customer_status string,
    created_at timestamp,
    updated_at timestamp
)
USING DELTA;

In [0]:
from pyspark.sql.functions import *
from delta.tables import *

df = spark.read.table("shopsphere.bronze.customers")

In [0]:
# trim spaces
df_trimmed = df.withColumn("first_name", trim(col("first_name")))
df_trimmed = df_trimmed.withColumn("last_name", trim(col("last_name")))
df_trimmed = df_trimmed.withColumn("email", trim(col("email")))
df_trimmed = df_trimmed.withColumn("phone", trim(col("phone")))
df_trimmed = df_trimmed.withColumn("city", trim(col("city")))
df_trimmed = df_trimmed.withColumn("state", trim(col("state")))
df_trimmed = df_trimmed.withColumn("country", trim(col("country")))
df_trimmed = df_trimmed.withColumn("customer_status", trim(col("customer_status")))

In [0]:
#drop duplicates
df_dodup = df_trimmed.dropDuplicates()

In [0]:
#lowercase email
df_lower = df_dodup.withColumn("email", lower(col("email")))


In [0]:
#fill NA
df_fillna = df_lower.fillna({"phone": "NA","email": "NA"})

In [0]:
#validate email
email_regex = r"^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}$"

df_valid = df_fillna.filter(col("email").rlike(email_regex))

df_quarantine = df_fillna.filter(col("email").rlike(email_regex)==False
                                ).withColumn("validation_status", lit("invalid_email"))


In [0]:
#quarantine rejected rows from email
quarantine_table = DeltaTable.forName(spark,"shopsphere.quarantine.customers")

quarantine_table.alias("target").merge(df_quarantine.alias("source"), \
                                       "target.customer_id = source.customer_id" ).\
                        whenNotMatchedInsertAll().\
                        execute()

In [0]:
#Standardize City/State
df_city = df_valid.withColumn("city", initcap(col("city")))

state_mapping = {"Telangana": "TS", 
                 "Karnataka": "KA", 
                 "Rajasthan": "RJ",
                 "Tamil Nadu": "TN", 
                 "Maharashtra": "MH", 
                 "Kerala": "KL",
                 "West Bengal": "WB", 
                 "Gujarat": "GJ", 
                 "Delhi": "DL",}

df_standardized = df_city.replace(state_mapping, subset=["state"])



In [0]:
#Validate customer status

expected_status = {"ACTIVE","INACTIVE"}

df_valid = df_standardized.filter(col("customer_status").isin(expected_status))

df_quarantine = df_standardized.filter(~col("customer_status").isin(expected_status)
                                      ).withColumn("validation_status", lit("invalid_customer_status"))

In [0]:
#quarantine rejected rows from customer status
quarantine_table = DeltaTable.forName(spark,"shopsphere.quarantine.customers")

quarantine_table.alias("target").merge(df_quarantine.alias("source"), \
                                       "target.customer_id = source.customer_id" ).\
                        whenNotMatchedInsertAll().\
                        execute()

In [0]:
#write transformed data to silver table
silver_table = DeltaTable.forName(spark,"shopsphere.silver.customers")

silver_table.alias("target").merge(df_valid.alias("source"), \
                                       "target.customer_id = source.customer_id" ).\
                        whenNotMatchedInsertAll().\
                        whenMatchedUpdateAll().\
                        execute()